# 06 - XGBoost Classifier

Train an XGBoost classifier to predict `performance_category`, then evaluate it on the same stratified 80/20 split used by the other model notebooks. The preprocessing pipeline is included in the saved artifact so the backend can accept the original feature columns. Outcome-derived fields are excluded to reduce target leakage.

This notebook also records accuracy, macro precision/recall, and macro-F1 in `models/model_metrics.json`. Macro-F1 gives each performance category equal weight, which is useful when the classes are imbalanced.

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'student_performance_processed.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
target = 'performance_category'
excluded = {'student_id', target, 'risk_level', 'final_marks', 'passed'}
features = [column for column in df.columns if column not in excluded]
X = df[features].copy()
y = df[target].astype(str)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Data shape: {df.shape}')
print(f'Feature count: {len(features)}')
print(f'Train rows: {len(X_train)} | Test rows: {len(X_test)}')
print('Class counts in training data:')
print(y_train.value_counts().sort_index())

In [ ]:
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]), categorical_features),
])

xgb_classifier = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,
    reg_lambda=1.0,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=2,
    verbosity=0,
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_classifier),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('XGBoost training complete.')

In [ ]:
metrics = {
    'model': 'XGBoost',
    'accuracy': round(accuracy_score(y_test, predictions), 4),
    'precision_macro': round(precision_score(y_test, predictions, average='macro', zero_division=0), 4),
    'recall_macro': round(recall_score(y_test, predictions, average='macro', zero_division=0), 4),
    'f1_macro': round(f1_score(y_test, predictions, average='macro', zero_division=0), 4),
}
print(metrics)
print('\nClassification report:')
print(classification_report(y_test, predictions, zero_division=0))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, predictions))

transformed_names = model.named_steps['preprocessor'].get_feature_names_out()
importance_values = model.named_steps['classifier'].feature_importances_
feature_importance = pd.DataFrame({
    'feature': transformed_names,
    'importance': importance_values,
}).sort_values('importance', ascending=False)
print('\nTop 15 XGBoost feature importances:')
display(feature_importance.head(15))

In [ ]:
model_path = MODEL_DIR / 'student_performance_xgboost.pkl'
with model_path.open('wb') as model_file:
    pickle.dump(model, model_file)

metrics_path = MODEL_DIR / 'model_metrics.json'
saved_metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else []
saved_metrics = [row for row in saved_metrics if row.get('model') != metrics['model']]
saved_metrics.append(metrics)
metrics_path.write_text(json.dumps(saved_metrics, indent=2))

print(f'Saved XGBoost pipeline: {model_path}')
print(f'Updated model comparison: {metrics_path}')

## Conclusion and Metric Guide

`n_estimators=300` means the boosted model builds 300 trees in sequence; it is not 300 epochs. `max_depth=4` limits each tree's depth, and `learning_rate=0.05` controls how much each new tree adjusts the current model. `subsample` and `colsample_bytree` add sampling to help control overfitting.

Accuracy is the share of test rows classified correctly. Macro precision, recall, and F1 compute each category's metric separately and average them equally, so smaller classes count too. Each probability in `predict_proba` is the model's estimated probability for a category; the largest probability determines the predicted category. Feature importance shows how much the fitted trees used each transformed feature for their splits; it is not a causal effect.

Compare these held-out results with notebooks 02–05 before deciding which model to deploy. The API selects the available model with the highest recorded macro-F1 after restart; XGBoost is available as `models/student_performance_xgboost.pkl` and its metrics appear in the frontend comparison.